# 07: Sentinel Service Verification & HITL Demo

This notebook demonstrates the **VeNRA Sentinel Service** in action. It validates an agent's response by:
1. Splitting the response into sentences.
2. Performing a "White-Box" groundedness check via the fine-tuned SLM Judge.
3. Aggregating scores for an overall reliability metric.

We will perform a live run against the Hugging Face hosted Judge and conduct a manual verification of its labels.

In [1]:
import requests
import json
import pandas as pd
from IPython.display import display, HTML

SENTINEL_URL = "http://localhost:8000/verify"
HF_JUDGE_URL = "https://pagand-venra-haldet.hf.space/verify"

## 0. Connectivity & Raw Response Pre-Check
Before testing the local Sentinel Service, we verify the Hugging Face Space and inspect the **RAW JSON** output to ensure key mapping is correct.

In [2]:
def test_hf_direct():
    payload = {
        "sentence": "Revenue for 2023 was $383 billion.", 
        "context": "Total net sales in 2023 were $383,285 million, compared to $394,328 million in 2022.", 
        "trace": "# Extracted revenue: 383e9"
    }
    print(f"Checking Hugging Face Judge at: {HF_JUDGE_URL}...")
    try:
        response = requests.post(HF_JUDGE_URL, json=payload, timeout=60)
        if response.status_code == 200:
            res_json = response.json()
            print("✅ Success! Hugging Face Judge is Online.")
            print("--- RAW JSON RESPONSE ---")
            print(json.dumps(res_json, indent=2))
            print("-------------------------")
            return True
        else:
            print(f"❌ Failed! Status: {response.status_code}")
            print(response.text)
            return False
    except Exception as e:
        print(f"❌ Connection Error: {e}")
        return False

hf_online = test_hf_direct()

Checking Hugging Face Judge at: https://pagand-venra-haldet.hf.space/verify...
✅ Success! Hugging Face Judge is Online.
--- RAW JSON RESPONSE ---
{
  "grounded": 0.6324470043182373,
  "common": 0.3655724823474884,
  "hallucination": 0.0019805459305644035,
  "prediction": "GROUNDED"
}
-------------------------


## 1. End-to-End Verification Run

Now we test the local Sentinel Service. Ensure you have restarted the server after the latest `sentinel.py` updates.

In [3]:
def verify_response(query, answer, context, trace):
    payload = {
        "query": query,
        "answer_text": answer,
        "context": context,
        "trace": trace
    }
    try:
        response = requests.post(SENTINEL_URL, json=payload, timeout=70)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Error: {response.status_code} - {response.text}")
            return None
    except Exception as e:
        print(f"Connection error to Sentinel Service: {e}")
        return None

query = "What was the 2023 revenue and what is the S&P 500?"
answer = "Revenue for 2023 was $383 billion. The S&P 500 is a stock market index tracking 500 large companies. The CEO personally guaranteed a 10% profit margin for 2024."
context = "Total net sales in 2023 were $383,285 million, compared to $394,328 million in 2022."
trace = "# Extracted revenue: 383e9\n# No calculation needed for S&P 500 statement."

if hf_online:
    result = verify_response(query, answer, context, trace)
    if result:
        print(f"Overall Groundedness Score: {result['overall_groundedness_score']:.2f}")
        df_res = pd.DataFrame(result['sentence_results'])
        cols = ['sentence', 'label', 'grounded_prob', 'common_prob', 'hallucination_prob', 'explanation']
        display(df_res[cols])
else:
    print("Skipping End-to-End test because Hugging Face Judge is offline.")

Overall Groundedness Score: 0.33


,sentence,label,grounded_prob,common_prob,hallucination_prob,explanation
0,Revenue for 2023 was $383 billion.,COMMON,0.201626,0.797279,0.001095,None
1,The S&P 500 is a stock market index tracking 5...,COMMON,0.030977,0.964862,0.004161,None
2,The CEO personally guaranteed a 10% profit mar...,GROUNDED,0.641013,0.244362,0.114625,None
